# Research V2: strict five-fold Jina-v2 boundary reranker

This notebook runs only the preregistered `CE -> parent max -> deterministic Top-5` hypothesis. It does not tune candidate retrieval, blend renderers, train LR, or upload a submission. By default it stops after writing the Fold-0 `PILOT_REPORT.json`, even when the pilot passes; full five-fold requires a separate manual post-review approval flag.

**First-time Kaggle setup.** Create one private Kaggle Dataset named `research-v2-jina-boundary` from the staged bundle. Attach it as an Input. In Notebook Settings choose GPU **T4 x2** and Persistence **Files only**. In Dependency Manager pin `transformers==4.40.2`, `peft==0.11.1`, `accelerate==0.30.1`, `safetensors==0.4.3`, and `einops==0.8.0`; do not upgrade NumPy/SciPy in-kernel. Outputs and resumable checkpoints are written under `/kaggle/working/research_v2_jina_boundary`. Save a Notebook Version before stopping. Stop safely with **Session -> Stop Session** only after both child processes have exited and the final manifest is present.

The two T4s are separate ~16 GiB devices. The notebook runs one fold process per GPU. Gradient accumulation increases effective batch size without changing per-microbatch VRAM and does not make gradients weaker: losses are divided by the accumulation count so the accumulated update is the mean gradient.

In [6]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, time
import torch, transformers, peft

matches = list(Path('/kaggle/input').rglob('KAGGLE_INPUT_MANIFEST.json'))
assert len(matches) == 1, f'Expected exactly one manifest, found: {matches}'

INPUT = matches[0].parent
WORK = Path('/kaggle/working/research_v2_jina_boundary')
WORK.mkdir(parents=True, exist_ok=True)

assert transformers.__version__.startswith('4.40.'), transformers.__version__
assert peft.__version__.startswith('0.11.'), peft.__version__
assert torch.cuda.device_count() == 2, f'Expected T4 x2, got {torch.cuda.device_count()}'

print([(i, torch.cuda.get_device_name(i),
        round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2))
       for i in range(2)])

print('Resolved INPUT:', INPUT)
print('WORK:', WORK)

[(0, 'Tesla T4', 14.56), (1, 'Tesla T4', 14.56)]
Resolved INPUT: /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary
WORK: /kaggle/working/research_v2_jina_boundary


In [7]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(8 << 20), b""):
            h.update(b)
    return h.hexdigest()

manifest = json.loads(
    (INPUT / "KAGGLE_INPUT_MANIFEST.json").read_text()
)

SKIP_FILES = {
    "RESEARCH_V2_JINA_BOUNDARY_KAGGLE.ipynb",
}

checked = 0
skipped = []

for rel, expected in manifest["files_sha256"].items():
    if rel in SKIP_FILES:
        skipped.append(rel)
        continue

    path = INPUT / rel
    assert path.exists(), f"Missing required input file: {rel}"

    actual = sha256(path)
    assert actual == expected, (rel, actual, expected)
    checked += 1

assert manifest["v2_folds_sha256"] == (
    "94ad5c6d5e582ced5eec8d2c3c15f938454c17e713614391091e72abea9aba19"
)

print(
    "INPUT CONTRACT PASS",
    f"checked={checked}",
    f"skipped={skipped}",
)

INPUT CONTRACT PASS checked=39 skipped=['RESEARCH_V2_JINA_BOUNDARY_KAGGLE.ipynb']


In [19]:
# =============================================================================
# KAGGLE COMPATIBILITY PATCH — Research V2 Jina-v2 boundary reranker
#
# NOTE FOR FUTURE AGENTS / GPT:
# The sealed Research V2 runner was validated in a different local PEFT /
# Transformers environment. Kaggle's pinned environment exposes three runtime
# compatibility differences in Jina-v2's custom remote code:
#   1) broad LoRA target "out_proj" collides with classifier.out_proj after
#      PEFT wraps the classifier in ModulesToSaveWrapper;
#   2) Jina-v2 does not implement HF get_input_embeddings(), so
#      enable_input_require_grads() fails under gradient checkpointing;
#   3) PEFT 0.11 LoRA Linear has no _cast_input_dtype() helper.
#
# This cell creates a Kaggle-only patched copy under /kaggle/working and runs
# a 1-step smoke test. It does NOT modify the sealed input bundle, training
# data, folds, loss, hyperparameters, model weights, or scientific contract.
# The intended contract remains:
# frozen Jina-v2 backbone + LoRA on 12 attention Wqkv/out_proj pairs +
# trainable classifier + gradient checkpointing.
# =============================================================================

from pathlib import Path
import os
import shutil
import subprocess
import sys

# -------------------------------------------------------------------------
# 0. Paths used by both smoke test and the real pilot
# -------------------------------------------------------------------------

PY = sys.executable

ORIGINAL_RUNNER = INPUT / "jina_v2_boundary_train.py"
RUNNER = WORK / "jina_v2_boundary_train_kaggle.py"

MODEL = INPUT / "jina-reranker-v2-base-multilingual"
GROUPS = INPUT / "V2_BOUNDARY_GROUPS.jsonl"
GROUP_MANIFEST = INPUT / "V2_BOUNDARY_GROUPS_MANIFEST.json"
POOL = INPUT / "V2_CANDIDATE_POOL.jsonl"
CONTEXTS = INPUT / "V2_CONTEXTS.jsonl"

# Kaggle SQLite compatibility:
# copy the sealed base-score DB byte-for-byte to /kaggle/working because
# SQLite read-only URI access can fail on Kaggle's dataset mount.

BASE_DB_INPUT = INPUT / "evidence_ab_scores.sqlite"
BASE_DB = WORK / "evidence_ab_scores.sqlite"

if not BASE_DB.exists() or sha256(BASE_DB) != sha256(BASE_DB_INPUT):
    shutil.copy2(BASE_DB_INPUT, BASE_DB)

assert sha256(BASE_DB) == sha256(BASE_DB_INPUT)

assert ORIGINAL_RUNNER.exists(), ORIGINAL_RUNNER
assert MODEL.exists(), MODEL
assert GROUPS.exists(), GROUPS
assert GROUP_MANIFEST.exists(), GROUP_MANIFEST

src = ORIGINAL_RUNNER.read_text(encoding="utf-8")


# -------------------------------------------------------------------------
# 1. PEFT target-module compatibility
#
# Original:
#   target_modules=["Wqkv", "out_proj"]
#
# Problem on PEFT 0.11:
# "out_proj" also matches classifier.out_proj after classifier has been
# wrapped by ModulesToSaveWrapper.
#
# Fix:
# enumerate exactly the 24 attention projections:
# 12 x Wqkv + 12 x attention out_proj.
# -------------------------------------------------------------------------

old_lora_config = '''    config = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=rank, lora_alpha=2 * rank,
        lora_dropout=0.05, target_modules=["Wqkv", "out_proj"],
        modules_to_save=["classifier"], bias="none")
    model = get_peft_model(model, config)'''

new_lora_config = '''    # Kaggle/PEFT compatibility:
    # target only transformer attention projections, never classifier.out_proj.
    backbone_targets = [
        name for name, _ in model.named_modules()
        if (
            name.endswith(".mixer.Wqkv")
            or name.endswith(".mixer.out_proj")
        )
        and name.startswith("roberta.encoder.layers.")
    ]

    if len(backbone_targets) != 24:
        raise RuntimeError(
            f"Expected exactly 24 Jina-v2 attention LoRA targets "
            f"(12 Wqkv + 12 out_proj), got {len(backbone_targets)}: "
            f"{backbone_targets}"
        )

    if any(name.startswith("classifier.") for name in backbone_targets):
        raise RuntimeError(
            f"Classifier leaked into LoRA targets: {backbone_targets}"
        )

    config = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=rank, lora_alpha=2 * rank,
        lora_dropout=0.05, target_modules=backbone_targets,
        modules_to_save=["classifier"], bias="none")
    model = get_peft_model(model, config)'''

assert old_lora_config in src, (
    "PATCH 1 FAILED: expected original LoRA config block was not found."
)

src = src.replace(old_lora_config, new_lora_config, 1)


# -------------------------------------------------------------------------
# 2. Gradient-checkpointing / input-grad compatibility
#
# Jina-v2 custom SequenceClassification wrapper does not implement
# get_input_embeddings(), therefore HF enable_input_require_grads() raises
# NotImplementedError.
#
# We attach the equivalent hook directly to the unique vocabulary embedding.
# Embedding weights remain frozen; only its OUTPUT requires gradients so
# checkpointed blocks can propagate gradients to LoRA parameters.
# -------------------------------------------------------------------------

old_input_grad = '''    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    return model.to("cuda"), tok'''

new_input_grad = '''    model.gradient_checkpointing_enable()

    # Kaggle/Jina-v2 compatibility:
    # custom remote-code wrapper does not implement get_input_embeddings().
    vocab_size = len(tok)
    embedding_candidates = [
        (name, module)
        for name, module in model.named_modules()
        if isinstance(module, torch.nn.Embedding)
        and module.num_embeddings == vocab_size
    ]

    if len(embedding_candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one vocabulary embedding with "
            f"num_embeddings={vocab_size}, got "
            f"{[(n, m.num_embeddings, m.embedding_dim) for n, m in embedding_candidates]}"
        )

    embedding_name, input_embedding = embedding_candidates[0]

    def _research_v2_make_embedding_output_require_grad(module, inputs, output):
        if not torch.is_tensor(output):
            raise RuntimeError(
                f"Expected tensor output from {embedding_name}, got {type(output)}"
            )
        output.requires_grad_(True)

    model._research_v2_input_grad_hook = (
        input_embedding.register_forward_hook(
            _research_v2_make_embedding_output_require_grad
        )
    )

    print(
        f"[Research V2] input-gradient hook attached to "
        f"{embedding_name} "
        f"(vocab={input_embedding.num_embeddings}, "
        f"dim={input_embedding.embedding_dim})"
    )

    return model.to("cuda"), tok'''

assert old_input_grad in src, (
    "PATCH 2 FAILED: expected input-gradient block was not found."
)

src = src.replace(old_input_grad, new_input_grad, 1)


# -------------------------------------------------------------------------
# 3. PEFT 0.11 tuple-LoRA dtype compatibility
#
# The runner's Jina-specific tuple-returning LoRA forward was written against
# PEFT versions exposing self._cast_input_dtype(). PEFT 0.11 does not.
#
# Preserve the same LoRA mathematics. Outside autocast, manually cast the
# LoRA branch input to lora_A dtype and cast its delta back to base dtype.
# -------------------------------------------------------------------------

old_dtype = '''            adapter_x = self._cast_input_dtype(x, lora_A.weight.dtype)
            projected = projected + lora_B(lora_A(dropout(adapter_x))) * scaling'''

new_dtype = '''            requires_conversion = not torch.is_autocast_enabled()

            if requires_conversion:
                expected_dtype = projected.dtype

                if hasattr(self, "_cast_input_dtype"):
                    adapter_x = self._cast_input_dtype(
                        x, lora_A.weight.dtype
                    )
                else:
                    # PEFT 0.11 compatibility.
                    adapter_x = x.to(lora_A.weight.dtype)
            else:
                adapter_x = x

            delta = lora_B(lora_A(dropout(adapter_x))) * scaling

            if requires_conversion:
                delta = delta.to(expected_dtype)

            projected = projected + delta'''

assert old_dtype in src, (
    "PATCH 3 FAILED: expected tuple-LoRA dtype block was not found."
)

src = src.replace(old_dtype, new_dtype, 1)


# -------------------------------------------------------------------------
# 4. Write a fresh patched runner to writable Kaggle storage
# -------------------------------------------------------------------------

RUNNER.parent.mkdir(parents=True, exist_ok=True)
RUNNER.write_text(src, encoding="utf-8")

assert RUNNER.exists()

print("Patched runner created:")
print(" ", RUNNER)
print("Patched runner SHA256:")
print(" ", sha256(RUNNER))


# -------------------------------------------------------------------------
# 5. One-step fail-closed smoke test
#
# This proves:
# model load -> PEFT injection -> input-grad hook -> forward -> backward ->
# optimizer step -> checkpoint path
#
# before launching the bounded 1,000-group / 400-step pilot.
# -------------------------------------------------------------------------

SMOKE = WORK / "lora_injection_smoke"

# Do not let a previous failed smoke leave misleading artifacts.
shutil.rmtree(SMOKE, ignore_errors=True)

smoke_cmd = [
    PY,
    str(RUNNER),
    "train",
    "--model", str(MODEL),
    "--groups", str(GROUPS),
    "--groups-manifest", str(GROUP_MANIFEST),
    "--fold", "fold_0",
    "--microbatch", "1",
    "--accumulation", "1",
    "--max-train-groups", "2",
    "--eval-steps", "1",
    "--output", str(SMOKE),
    "--max-steps", "1",
]

smoke_env = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="0",
    TOKENIZERS_PARALLELISM="false",
    HF_MODULES_CACHE=str(WORK / "hf_modules_gpu0"),
)

print("\nSMOKE COMMAND:")
print(" ".join(smoke_cmd))
print("\n===== SMOKE OUTPUT =====\n")

result = subprocess.run(
    smoke_cmd,
    env=smoke_env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)
print("RETURN CODE:", result.returncode)

assert result.returncode == 0, (
    "Research V2 Kaggle compatibility smoke test FAILED. "
    "Read the traceback above; do not launch the bounded pilot."
)

print("\n============================================================")
print("KAGGLE COMPATIBILITY SMOKE PASS")
print("Use this RUNNER for the bounded strict-OOF pilot:")
print(RUNNER)
print("============================================================")

Patched runner created:
  /kaggle/working/research_v2_jina_boundary/jina_v2_boundary_train_kaggle.py
Patched runner SHA256:
  e873e49175b5bdf37ab85ae9349e6b6d847982ab4abb3a63fe66e3f2225f3fb2

SMOKE COMMAND:
/usr/bin/python3 /kaggle/working/research_v2_jina_boundary/jina_v2_boundary_train_kaggle.py train --model /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/jina-reranker-v2-base-multilingual --groups /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/V2_BOUNDARY_GROUPS.jsonl --groups-manifest /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/V2_BOUNDARY_GROUPS_MANIFEST.json --fold fold_0 --microbatch 1 --accumulation 1 --max-train-groups 2 --eval-steps 1 --output /kaggle/working/research_v2_jina_boundary/lora_injection_smoke --max-steps 1

===== SMOKE OUTPUT =====

flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is

## Bounded strict-OOF pilot

Fold 0 is trained only from folds 1-4 (with cross-fold exact/near-duplicate exclusions) and scored only on fold 0. The fixed pilot is 1,000 training groups and at most 400 optimizer microsteps. PASS requires Recall delta >= 0.003, improved boundary-pair accuracy, wins > losses, and multi-gold delta >= -0.005. Failure stops the family; no grid is attempted.

In [20]:
renderer=manifest['renderer']
probe=WORK/'resume_probe'
probe_env=dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', HF_MODULES_CACHE=str(WORK/'hf_modules_gpu0'))
probe_common=[PY,str(RUNNER),'train','--model',str(MODEL),'--groups',str(GROUPS),'--groups-manifest',str(GROUP_MANIFEST),'--fold','fold_0','--microbatch','1','--accumulation','1','--max-train-groups','2','--eval-steps','1']
subprocess.run(probe_common+['--output',str(probe/'continuous'),'--max-steps','2'],env=probe_env,check=True)
subprocess.run(probe_common+['--output',str(probe/'first'),'--max-steps','1'],env=probe_env,check=True)
subprocess.run(probe_common+['--output',str(probe/'resumed'),'--max-steps','2','--resume',str(probe/'first'/'checkpoint-000001')],env=probe_env,check=True)
from safetensors.torch import load_file
a=load_file(probe/'continuous'/'checkpoint-000002'/'adapter'/'adapter_model.safetensors')
b=load_file(probe/'resumed'/'checkpoint-000002'/'adapter'/'adapter_model.safetensors')
resume_parity=(a.keys()==b.keys() and all(torch.equal(a[k],b[k]) for k in a))
assert resume_parity, 'Checkpoint/resume tensor parity failed'
pilot=WORK/'pilot_fold_0'
env=dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', HF_MODULES_CACHE=str(WORK/'hf_modules_gpu0'))
train_cmd=[PY,str(RUNNER),'train','--model',str(MODEL),'--groups',str(GROUPS),'--groups-manifest',str(GROUP_MANIFEST),'--fold','fold_0','--output',str(pilot),'--microbatch','2','--accumulation','8','--max-train-groups','1000','--max-steps','400','--eval-steps','96']
subprocess.run(train_cmd,env=env,check=True)
score_cmd=[PY,str(RUNNER),'score','--model',str(MODEL),'--groups',str(GROUPS),'--fold','fold_0','--adapter',str(pilot/'best'/'adapter'),'--pool',str(POOL),'--contexts-pack',str(CONTEXTS),'--base-score-db',str(BASE_DB),'--renderer',renderer,'--output',str(pilot/'score'),'--score-batch','16']
subprocess.run(score_cmd,env=env,check=True)
m=json.loads((pilot/'score'/'fold_0_METRICS.json').read_text())
multi_delta=m['multi_gold']['ft_recall']-m['multi_gold']['base_recall']
gate_checks={'delta_recall_at_5_gte_0_003':m['delta_recall_at_5']>=0.003,'boundary_pair_accuracy_improved':m['boundary_pair_accuracy']['ft']>m['boundary_pair_accuracy']['base'],'wins_exceed_losses':m['wins']>m['losses'],'multi_gold_delta_gte_minus_0_005':multi_delta>=-0.005}
train_manifest=json.loads((pilot/'TRAINING_MANIFEST.json').read_text())
gate_checks['peak_allocated_vram_mib_lt_15000']=train_manifest['peak_allocated_mib']<15000
gate_checks['resume_tensor_parity']=resume_parity
pilot_pass=all(gate_checks.values())
prediction_path=pilot/'score'/'fold_0_PREDICTIONS.jsonl'
metrics_path=pilot/'score'/'fold_0_METRICS.json'
continuous_adapter=probe/'continuous'/'checkpoint-000002'/'adapter'/'adapter_model.safetensors'
resumed_adapter=probe/'resumed'/'checkpoint-000002'/'adapter'/'adapter_model.safetensors'
pilot_report={'schema_version':'dsc2026.research_v2.jina_boundary_pilot_report.v1','status':'PASS' if pilot_pass else 'FAIL_STOP_FAMILY','next_action':'STOP_FOR_USER_REVIEW' if pilot_pass else 'STOP_FAMILY_WITHOUT_GRID','full_five_fold_launched':False,'immutable_contract':{'fold':'fold_0','candidate_pool_sha256':sha256(POOL),'groups_sha256':sha256(GROUPS),'groups_manifest_sha256':sha256(GROUP_MANIFEST),'renderer':renderer,'base_model_directory':str(MODEL),'max_length':512,'parent_aggregation':'max','output':'deterministic Top-5; no LR, rules, or fusion'},'pilot_config':{'max_train_groups':1000,'max_steps':400,'microbatch_parent_pairs':2,'gradient_accumulation':8,'eval_steps':96},'preregistered_gate':{'delta_recall_at_5_min':0.003,'boundary_pair_accuracy_must_improve':True,'wins_must_exceed_losses':True,'multi_gold_delta_min':-0.005,'peak_allocated_vram_mib_max_exclusive':15000,'resume_tensor_parity_required':True},'gate_checks':gate_checks,'multi_gold_delta':multi_delta,'metrics':m,'training':train_manifest,'resume_probe':{'tensor_parity':resume_parity,'continuous_adapter_sha256':sha256(continuous_adapter),'resumed_adapter_sha256':sha256(resumed_adapter)},'environment':{'python':sys.version,'torch':torch.__version__,'transformers':transformers.__version__,'peft':peft.__version__,'gpus':[{'index':i,'name':torch.cuda.get_device_name(i),'total_memory_bytes':torch.cuda.get_device_properties(i).total_memory} for i in range(torch.cuda.device_count())]},'artifacts':{'input_manifest_sha256':sha256(INPUT/'KAGGLE_INPUT_MANIFEST.json'),'runner_sha256':sha256(RUNNER),'fold0_predictions':str(prediction_path),'fold0_predictions_sha256':sha256(prediction_path),'fold0_metrics':str(metrics_path),'fold0_metrics_sha256':sha256(metrics_path),'training_manifest_sha256':sha256(pilot/'TRAINING_MANIFEST.json')}}
(WORK/'PILOT_REPORT.json').write_text(json.dumps(pilot_report,indent=2))
(WORK/'PILOT_GATE.json').write_text(json.dumps({'status':pilot_report['status'],'report':'PILOT_REPORT.json','full_five_fold_launched':False},indent=2))
print(json.dumps(pilot_report,indent=2))
print('DEFAULT STOP: review PILOT_REPORT.json before enabling the separate full-five-fold cell.')

flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
You are usin

[Research V2] input-gradient hook attached to base_model.model.roberta.embeddings.word_embeddings (vocab=250002, dim=768)
{"step": 1, "validation_pair_accuracy": 0.8475, "best_validation_pair_accuracy": 0.8475, "bad_checks": 0, "train_pairs": 8, "valid_pairs": 2284, "loss": 0.3802987337112427}


/kaggle/working/research_v2_jina_boundary/jina_v2_boundary_train_kaggle.py:413: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(json.dumps({**state, "loss": float(loss) * args.accumulation}), flush=True)
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/jina-reranker-v2-base-multilingual - will assume that the vocabulary was not modified.
  warnings.warn(


{"step": 2, "validation_pair_accuracy": 0.8475, "best_validation_pair_accuracy": 0.8475, "bad_checks": 1, "train_pairs": 8, "valid_pairs": 2284, "loss": 0.33655089139938354}
{
  "schema_version": "dsc2026.research_v2.jina_boundary_checkpoint.v1",
  "status": "COMPLETE",
  "held_fold": "fold_0",
  "base_model": "/kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/jina-reranker-v2-base-multilingual",
  "renderer": "lexical",
  "train_groups": 2,
  "valid_groups": 522,
  "forbidden_duplicate_qids": [
    "114846",
    "117908",
    "139536",
    "156640",
    "61406",
    "63562",
    "77610"
  ],
  "pairwise_parent_max_loss": true,
  "max_length": 512,
  "microbatch_parent_pairs": 1,
  "gradient_accumulation": 1,
  "lora": {
    "rank": 8,
    "targets": [
      "Wqkv",
      "out_proj"
    ],
    "classifier_trainable": true
  },
  "runtime_seconds": 34.40776576899998,
  "peak_allocated_mib": 760.033203125,
  "groups_sha256": "681ca1340dac9e6b498cbde30fa8ac7fee98840664a

flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
You are usin

...

[Research V2] input-gradient hook attached to base_model.model.roberta.embeddings.word_embeddings (vocab=250002, dim=768)
{"step": 96, "validation_pair_accuracy": 0.855, "best_validation_pair_accuracy": 0.855, "bad_checks": 0, "train_pairs": 4332, "valid_pairs": 2284, "loss": 0.7898985743522644}
{"step": 192, "validation_pair_accuracy": 0.8525, "best_validation_pair_accuracy": 0.855, "bad_checks": 1, "train_pairs": 4332, "valid_pairs": 2284, "loss": 0.7334250211715698}
{"step": 288, "validation_pair_accuracy": 0.8525, "best_validation_pair_accuracy": 0.855, "bad_checks": 2, "train_pairs": 4332, "valid_pairs": 2284, "loss": 0.1515987664461136}
{
  "schema_version": "dsc2026.research_v2.jina_boundary_checkpoint.v1",
  "status": "COMPLETE",
  "held_fold": "fold_0",
  "base_model": "/kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/jina-reranker-v2-base-multilingual",
  "renderer": "lexical",
  "train_groups": 1000,
  "valid_groups": 522,
  "forbidden_duplicate_qids": [


flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
You are usin

[Research V2] input-gradient hook attached to base_model.model.roberta.embeddings.word_embeddings (vocab=250002, dim=768)


Traceback (most recent call last):
  File "/kaggle/working/research_v2_jina_boundary/jina_v2_boundary_train_kaggle.py", line 560, in <module>
    train(args) if args.stage == "train" else score(args)
                                              ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/research_v2_jina_boundary/jina_v2_boundary_train_kaggle.py", line 448, in score
    base.execute("SELECT qid,doc_id,score FROM scores WHERE arm=?", (args.renderer,))}
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
sqlite3.OperationalError: unable to open database file


CalledProcessError: Command '['/usr/bin/python3', '/kaggle/working/research_v2_jina_boundary/jina_v2_boundary_train_kaggle.py', 'score', '--model', '/kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/jina-reranker-v2-base-multilingual', '--groups', '/kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/V2_BOUNDARY_GROUPS.jsonl', '--fold', 'fold_0', '--adapter', '/kaggle/working/research_v2_jina_boundary/pilot_fold_0/best/adapter', '--pool', '/kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/V2_CANDIDATE_POOL.jsonl', '--contexts-pack', '/kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/V2_CONTEXTS.jsonl', '--base-score-db', '/kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/evidence_ab_scores.sqlite', '--renderer', 'lexical', '--output', '/kaggle/working/research_v2_jina_boundary/pilot_fold_0/score', '--score-batch', '16']' returned non-zero exit status 1.

In [21]:
import shutil
import sqlite3
from pathlib import Path

# ------------------------------------------------------------------
# Kaggle SQLite compatibility:
# Keep the sealed DB immutable in /kaggle/input, but make a byte-identical
# working copy because SQLite can fail to open databases directly from
# Kaggle's read-only dataset mount.
# ------------------------------------------------------------------

BASE_DB_INPUT = INPUT / "evidence_ab_scores.sqlite"
BASE_DB = WORK / "evidence_ab_scores.sqlite"

assert BASE_DB_INPUT.exists(), f"Missing source DB: {BASE_DB_INPUT}"

# Copy only if absent or hash differs.
if (
    not BASE_DB.exists()
    or sha256(BASE_DB) != sha256(BASE_DB_INPUT)
):
    shutil.copy2(BASE_DB_INPUT, BASE_DB)

assert BASE_DB.exists()
assert sha256(BASE_DB) == sha256(BASE_DB_INPUT), (
    "SQLite working copy does not match sealed input DB"
)

print("SQLite source :", BASE_DB_INPUT)
print("SQLite working:", BASE_DB)
print("SHA256        :", sha256(BASE_DB))
print("Size bytes    :", BASE_DB.stat().st_size)

# Fail early: prove SQLite can actually open/query the local copy.
con = sqlite3.connect(str(BASE_DB))
try:
    print("integrity_check:", con.execute("PRAGMA integrity_check").fetchone()[0])
    print(
        "score rows:",
        con.execute("SELECT COUNT(*) FROM scores").fetchone()[0]
    )
    print(
        "arms:",
        con.execute(
            "SELECT arm, COUNT(*) FROM scores GROUP BY arm ORDER BY arm"
        ).fetchall()
    )
finally:
    con.close()

print("LOCAL SQLITE PREFLIGHT PASS")

SQLite source : /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/evidence_ab_scores.sqlite
SQLite working: /kaggle/working/research_v2_jina_boundary/evidence_ab_scores.sqlite
SHA256        : e2b00064a5bcce90cf6558027ff90c30aeb3baa67a78cb97ea81f5445eb95f7f
Size bytes    : 56856576
integrity_check: ok
score rows: 730900
arms: [('lexical', 365450), ('structural', 365450)]
LOCAL SQLITE PREFLIGHT PASS


In [22]:
pilot = WORK / "pilot_fold_0"

assert (pilot / "best" / "adapter").exists(), (
    "Pilot adapter missing; do not continue."
)

score_cmd = [
    PY, str(RUNNER),
    "score",
    "--model", str(MODEL),
    "--groups", str(GROUPS),
    "--fold", "fold_0",
    "--adapter", str(pilot / "best" / "adapter"),
    "--pool", str(POOL),
    "--contexts-pack", str(CONTEXTS),
    "--base-score-db", str(BASE_DB),   # <- working copy, NOT /kaggle/input
    "--renderer", renderer,
    "--output", str(pilot / "score"),
    "--score-batch", "16",
]

print(" ".join(score_cmd))

subprocess.run(
    score_cmd,
    env=env,
    check=True,
)

print("SCORING PASS")

/usr/bin/python3 /kaggle/working/research_v2_jina_boundary/jina_v2_boundary_train_kaggle.py score --model /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/jina-reranker-v2-base-multilingual --groups /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/V2_BOUNDARY_GROUPS.jsonl --fold fold_0 --adapter /kaggle/working/research_v2_jina_boundary/pilot_fold_0/best/adapter --pool /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/V2_CANDIDATE_POOL.jsonl --contexts-pack /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/V2_CONTEXTS.jsonl --base-score-db /kaggle/working/research_v2_jina_boundary/evidence_ab_scores.sqlite --renderer lexical --output /kaggle/working/research_v2_jina_boundary/pilot_fold_0/score --score-batch 16


flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
You are usin

[Research V2] input-gradient hook attached to base_model.model.roberta.embeddings.word_embeddings (vocab=250002, dim=768)
scored=100
scored=200
scored=300
scored=400
scored=500
scored=600
scored=700
scored=800
scored=900
scored=1000
scored=1100
scored=1200
scored=1300
{
  "schema_version": "dsc2026.research_v2.jina_boundary_fold_metric.v1",
  "status": "COMPLETE",
  "fold": "fold_0",
  "queries": 1398,
  "base_recall_at_5": 0.8670720076299474,
  "ft_recall_at_5": 0.856700047687172,
  "base_precision_at_5": 0.18440629470672387,
  "ft_precision_at_5": 0.18197424892703862,
  "delta_recall_at_5": -0.010371959942775394,
  "wins": 25,
  "losses": 41,
  "ties": 1332,
  "changed_top5_sets": 981,
  "single_gold": {
    "queries": 1288,
    "base_recall": 0.8874223602484472,
    "ft_recall": 0.8781055900621118
  },
  "multi_gold": {
    "queries": 110,
    "base_recall": 0.6287878787878787,
    "ft_recall": 0.606060606060606
  },
  "boundary_pair_accuracy": {
    "base": 0.7476543209876543,
    

In [24]:
m=json.loads((pilot/'score'/'fold_0_METRICS.json').read_text())
multi_delta=m['multi_gold']['ft_recall']-m['multi_gold']['base_recall']
gate_checks={'delta_recall_at_5_gte_0_003':m['delta_recall_at_5']>=0.003,'boundary_pair_accuracy_improved':m['boundary_pair_accuracy']['ft']>m['boundary_pair_accuracy']['base'],'wins_exceed_losses':m['wins']>m['losses'],'multi_gold_delta_gte_minus_0_005':multi_delta>=-0.005}
train_manifest=json.loads((pilot/'TRAINING_MANIFEST.json').read_text())
gate_checks['peak_allocated_vram_mib_lt_15000']=train_manifest['peak_allocated_mib']<15000
gate_checks['resume_tensor_parity']=resume_parity
pilot_pass=all(gate_checks.values())
prediction_path=pilot/'score'/'fold_0_PREDICTIONS.jsonl'
metrics_path=pilot/'score'/'fold_0_METRICS.json'
continuous_adapter=probe/'continuous'/'checkpoint-000002'/'adapter'/'adapter_model.safetensors'
resumed_adapter=probe/'resumed'/'checkpoint-000002'/'adapter'/'adapter_model.safetensors'
pilot_report={'schema_version':'dsc2026.research_v2.jina_boundary_pilot_report.v1','status':'PASS' if pilot_pass else 'FAIL_STOP_FAMILY','next_action':'STOP_FOR_USER_REVIEW' if pilot_pass else 'STOP_FAMILY_WITHOUT_GRID','full_five_fold_launched':False,'immutable_contract':{'fold':'fold_0','candidate_pool_sha256':sha256(POOL),'groups_sha256':sha256(GROUPS),'groups_manifest_sha256':sha256(GROUP_MANIFEST),'renderer':renderer,'base_model_directory':str(MODEL),'max_length':512,'parent_aggregation':'max','output':'deterministic Top-5; no LR, rules, or fusion'},'pilot_config':{'max_train_groups':1000,'max_steps':400,'microbatch_parent_pairs':2,'gradient_accumulation':8,'eval_steps':96},'preregistered_gate':{'delta_recall_at_5_min':0.003,'boundary_pair_accuracy_must_improve':True,'wins_must_exceed_losses':True,'multi_gold_delta_min':-0.005,'peak_allocated_vram_mib_max_exclusive':15000,'resume_tensor_parity_required':True},'gate_checks':gate_checks,'multi_gold_delta':multi_delta,'metrics':m,'training':train_manifest,'resume_probe':{'tensor_parity':resume_parity,'continuous_adapter_sha256':sha256(continuous_adapter),'resumed_adapter_sha256':sha256(resumed_adapter)},'environment':{'python':sys.version,'torch':torch.__version__,'transformers':transformers.__version__,'peft':peft.__version__,'gpus':[{'index':i,'name':torch.cuda.get_device_name(i),'total_memory_bytes':torch.cuda.get_device_properties(i).total_memory} for i in range(torch.cuda.device_count())]},'artifacts':{'input_manifest_sha256':sha256(INPUT/'KAGGLE_INPUT_MANIFEST.json'),'runner_sha256':sha256(RUNNER),'fold0_predictions':str(prediction_path),'fold0_predictions_sha256':sha256(prediction_path),'fold0_metrics':str(metrics_path),'fold0_metrics_sha256':sha256(metrics_path),'training_manifest_sha256':sha256(pilot/'TRAINING_MANIFEST.json')}}
(WORK/'PILOT_REPORT.json').write_text(json.dumps(pilot_report,indent=2))
(WORK/'PILOT_GATE.json').write_text(json.dumps({'status':pilot_report['status'],'report':'PILOT_REPORT.json','full_five_fold_launched':False},indent=2))
print(json.dumps(pilot_report,indent=2))
print('DEFAULT STOP: review PILOT_REPORT.json before enabling the separate full-five-fold cell.')

{
  "schema_version": "dsc2026.research_v2.jina_boundary_pilot_report.v1",
  "status": "FAIL_STOP_FAMILY",
  "next_action": "STOP_FAMILY_WITHOUT_GRID",
  "full_five_fold_launched": false,
  "immutable_contract": {
    "fold": "fold_0",
    "candidate_pool_sha256": "96a44e66549cc211e1f9d0fabb84fc825db3f21f32d5b349eeca3b1c0413e277",
    "groups_sha256": "681ca1340dac9e6b498cbde30fa8ac7fee98840664ae51a126491d9173009b2f",
    "groups_manifest_sha256": "4a6270b9f3e0f9f7f260443107b9868d3fcf634de6a8c0d93e32a20d95a59644",
    "renderer": "lexical",
    "base_model_directory": "/kaggle/input/datasets/hoangnguyenkaggle62/research-v2-jina-boundary/jina-reranker-v2-base-multilingual",
    "max_length": 512,
    "parent_aggregation": "max",
    "output": "deterministic Top-5; no LR, rules, or fusion"
  },
  "pilot_config": {
    "max_train_groups": 1000,
    "max_steps": 400,
    "microbatch_parent_pairs": 2,
    "gradient_accumulation": 8,
    "eval_steps": 96
  },
  "preregistered_gate": {
    "d

## Full five-fold campaign (manual launch only after pilot review)

The notebook stops after Fold-0 even when the pilot passes. Review `PILOT_REPORT.json` first, then deliberately set `USER_REVIEWED_PILOT_AND_APPROVES_FULL=True` in the next cell to launch full five-fold. Leaving the flag at its default `False` raises the expected fail-closed stop before any fold process is created. Each fold starts from the same frozen base and has its own adapter. Two child processes run in parallel, one per T4. If a session ends, set `RESUME` entries below to the last complete `checkpoint-*` directories and rerun that wave; the runner restores adapter, optimizer, and step.

In [ ]:
USER_REVIEWED_PILOT_AND_APPROVES_FULL=False  # change manually only after reviewing PILOT_REPORT.json
if not USER_REVIEWED_PILOT_AND_APPROVES_FULL:
    raise RuntimeError('EXPECTED DEFAULT STOP: full five-fold requires explicit user review approval')
pilot_report=json.loads((WORK/'PILOT_REPORT.json').read_text())
assert pilot_report['status']=='PASS', 'Full five-fold is forbidden because the fixed pilot did not pass.'
assert pilot_report['full_five_fold_launched'] is False
pilot_report['full_five_fold_launched']=True
pilot_report['full_launch_authorization']='USER_REVIEWED_PILOT_AND_APPROVES_FULL=True'
(WORK/'PILOT_REPORT.json').write_text(json.dumps(pilot_report,indent=2))
FULL=WORK/'full'; FULL.mkdir(exist_ok=True)
RESUME={}  # e.g. {'fold_2': FULL/'fold_2'/'checkpoint-001000'}
pilot_train=json.loads((pilot/'TRAINING_MANIFEST.json').read_text())
microbatch=4 if pilot_train['peak_allocated_mib']<7500 else 2
accumulation=4 if microbatch==4 else 8
def launch_train(fold,gpu):
    out=FULL/fold; out.mkdir(exist_ok=True)
    cmd=[PY,str(RUNNER),'train','--model',str(MODEL),'--groups',str(GROUPS),'--groups-manifest',str(GROUP_MANIFEST),'--fold',fold,'--output',str(out),'--microbatch',str(microbatch),'--accumulation',str(accumulation),'--eval-steps','200']
    if fold in RESUME: cmd += ['--resume',str(RESUME[fold])]
    log=open(out/'train.log','a',buffering=1)
    return subprocess.Popen(cmd,env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),TOKENIZERS_PARALLELISM='false',HF_MODULES_CACHE=str(WORK/f'hf_modules_gpu{gpu}')),stdout=log,stderr=subprocess.STDOUT),log
for wave in [('fold_0','fold_1'),('fold_2','fold_3'),('fold_4',)]:
    jobs=[launch_train(f,g) for g,f in enumerate(wave)]
    codes=[p.wait() for p,_ in jobs]
    for _,log in jobs: log.close()
    assert all(c==0 for c in codes),(wave,codes)
print('all fold checkpoints complete')

In [ ]:
if not globals().get('USER_REVIEWED_PILOT_AND_APPROVES_FULL',False):
    raise RuntimeError('EXPECTED DEFAULT STOP: full five-fold scoring also requires explicit user review approval')
assert json.loads((WORK/'PILOT_REPORT.json').read_text())['full_five_fold_launched'] is True
def launch_score(fold,gpu):
    out=FULL/fold/'score'; out.mkdir(exist_ok=True)
    cmd=[PY,str(RUNNER),'score','--model',str(MODEL),'--groups',str(GROUPS),'--fold',fold,'--adapter',str(FULL/fold/'best'/'adapter'),'--pool',str(POOL),'--contexts-pack',str(CONTEXTS),'--base-score-db',str(BASE_DB),'--renderer',renderer,'--output',str(out),'--score-batch','16']
    log=open(out/'score.log','a',buffering=1)
    return subprocess.Popen(cmd,env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),TOKENIZERS_PARALLELISM='false',HF_MODULES_CACHE=str(WORK/f'hf_modules_gpu{gpu}')),stdout=log,stderr=subprocess.STDOUT),log
for wave in [('fold_0','fold_1'),('fold_2','fold_3'),('fold_4',)]:
    jobs=[launch_score(f,g) for g,f in enumerate(wave)]
    codes=[p.wait() for p,_ in jobs]
    for _,log in jobs: log.close()
    assert all(c==0 for c in codes),(wave,codes)
metrics=[json.loads((FULL/f/'score'/f'{f}_METRICS.json').read_text()) for f in [f'fold_{i}' for i in range(5)]]
n=sum(m['queries'] for m in metrics)
def pooled(field): return sum(m[field]*m['queries'] for m in metrics)/n
def pooled_slice(name,field):
    z=sum(m[name]['queries'] for m in metrics); return sum(m[name][field]*m[name]['queries'] for m in metrics)/z
summary={'status':'COMPLETE_STRICT_FIVE_FOLD','queries':n,'pooled_base_recall_at_5':pooled('base_recall_at_5'),'pooled_ft_recall_at_5':pooled('ft_recall_at_5'),'pooled_base_precision_at_5':pooled('base_precision_at_5'),'pooled_ft_precision_at_5':pooled('ft_precision_at_5'),'single_gold':{'base':pooled_slice('single_gold','base_recall'),'ft':pooled_slice('single_gold','ft_recall')},'multi_gold':{'base':pooled_slice('multi_gold','base_recall'),'ft':pooled_slice('multi_gold','ft_recall')},'wins':sum(m['wins'] for m in metrics),'losses':sum(m['losses'] for m in metrics),'ties':sum(m['ties'] for m in metrics),'changed_top5_sets':sum(m['changed_top5_sets'] for m in metrics),'peak_allocated_mib_per_fold':{m['fold']:m['peak_allocated_mib'] for m in metrics},'runtime_seconds_per_fold':{m['fold']:m['runtime_seconds'] for m in metrics},'per_fold':metrics}
summary['pooled_delta_recall_at_5']=summary['pooled_ft_recall_at_5']-summary['pooled_base_recall_at_5']
(WORK/'FULL_OOF_SUMMARY.json').write_text(json.dumps(summary,indent=2))
print(json.dumps({k:v for k,v in summary.items() if k!='per_fold'},indent=2))